In [3]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

In [5]:
PROJECT_ROOT = Path("..").resolve()

RAW_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "jena"
    / "jena_climate_2009_2016.xlsx"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PARQUET_FILE = (
    PROCESSED_DIR
    / "observations.parquet"
)

QUALITY_REPORT_FILE = (
    PROCESSED_DIR
    / "data_quality_report.json"
)

STATION_METADATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "station_metadata.csv"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw file:")
print(RAW_FILE)

print("\nRaw file exists:")
print(RAW_FILE.exists())

Project root:
C:\Anomaly_Detection

Raw file:
C:\Anomaly_Detection\data\raw\jena\jena_climate_2009_2016.xlsx

Raw file exists:
True


In [6]:
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Processed directory ready:")
print(PROCESSED_DIR)

Processed directory ready:
C:\Anomaly_Detection\data\processed


In [7]:
raw_df = pd.read_excel(
    RAW_FILE,
    engine="openpyxl"
)

print("Dataset loaded successfully.")

print("\nShape:")
print(raw_df.shape)

print("\nColumns:")
print(raw_df.columns.tolist())

Dataset loaded successfully.

Shape:
(420551, 15)

Columns:
['Date Time', 'p (mbar)', 'T (degC)', 'Tpot (K)', 'Tdew (degC)', 'rh (%)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)', 'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)', 'wd (deg)']


In [9]:
raw_df.head()
raw_df.dtypes

Date Time              str
p (mbar)           float64
T (degC)           float64
Tpot (K)           float64
Tdew (degC)        float64
rh (%)             float64
VPmax (mbar)       float64
VPact (mbar)       float64
VPdef (mbar)       float64
sh (g/kg)          float64
H2OC (mmol/mol)    float64
rho (g/m**3)       float64
wv (m/s)           float64
max. wv (m/s)      float64
wd (deg)           float64
dtype: object

In [13]:
raw_missing = raw_df.isna().sum()

raw_missing

raw_missing_percentage = (
    raw_df.isna().mean() * 100
).round(4)

raw_missing_percentage

Date Time          0.0
p (mbar)           0.0
T (degC)           0.0
Tpot (K)           0.0
Tdew (degC)        0.0
rh (%)             0.0
VPmax (mbar)       0.0
VPact (mbar)       0.0
VPdef (mbar)       0.0
sh (g/kg)          0.0
H2OC (mmol/mol)    0.0
rho (g/m**3)       0.0
wv (m/s)           0.0
max. wv (m/s)      0.0
wd (deg)           0.0
dtype: float64

In [14]:
required_source_columns = [
    "Date Time",
    "p (mbar)",
    "T (degC)",
    "rh (%)"
]

missing_source_columns = [
    column
    for column in required_source_columns
    if column not in raw_df.columns
]

if missing_source_columns:
    raise ValueError(
        f"Required columns missing from workbook: "
        f"{missing_source_columns}"
    )

df = raw_df[
    required_source_columns
].copy()

df.head()

,Date Time,p (mbar),T (degC),rh (%)
0,01.01.2009 00:10:00,996.52,-8.02,93.3
1,01.01.2009 00:20:00,996.57,-8.41,93.4
2,01.01.2009 00:30:00,996.53,-8.51,93.9
3,01.01.2009 00:40:00,996.51,-8.31,94.2
4,01.01.2009 00:50:00,996.51,-8.27,94.1


In [15]:
df = df.rename(
    columns={
        "Date Time": "timestamp",
        "p (mbar)": "pressure_hpa",
        "T (degC)": "temperature_c",
        "rh (%)": "relative_humidity_pct"
    }
)

df.head()

,timestamp,pressure_hpa,temperature_c,relative_humidity_pct
0,01.01.2009 00:10:00,996.52,-8.02,93.3
1,01.01.2009 00:20:00,996.57,-8.41,93.4
2,01.01.2009 00:30:00,996.53,-8.51,93.9
3,01.01.2009 00:40:00,996.51,-8.31,94.2
4,01.01.2009 00:50:00,996.51,-8.27,94.1


In [16]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="%d.%m.%Y %H:%M:%S",
    errors="coerce"
)

print("Invalid timestamps:")
print(df["timestamp"].isna().sum())

df.head()

Invalid timestamps:
0


,timestamp,pressure_hpa,temperature_c,relative_humidity_pct
0,2009-01-01 00:10:00,996.52,-8.02,93.3
1,2009-01-01 00:20:00,996.57,-8.41,93.4
2,2009-01-01 00:30:00,996.53,-8.51,93.9
3,2009-01-01 00:40:00,996.51,-8.31,94.2
4,2009-01-01 00:50:00,996.51,-8.27,94.1


In [17]:
sensor_columns = [
    "temperature_c",
    "pressure_hpa",
    "relative_humidity_pct"
]

for column in sensor_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

df.dtypes

timestamp                datetime64[us]
pressure_hpa                    float64
temperature_c                   float64
relative_humidity_pct           float64
dtype: object

In [18]:
df.insert(
    1,
    "station_id",
    "JENA_001"
)

df.head()

,timestamp,station_id,pressure_hpa,temperature_c,relative_humidity_pct
0,2009-01-01 00:10:00,JENA_001,996.52,-8.02,93.3
1,2009-01-01 00:20:00,JENA_001,996.57,-8.41,93.4
2,2009-01-01 00:30:00,JENA_001,996.53,-8.51,93.9
3,2009-01-01 00:40:00,JENA_001,996.51,-8.31,94.2
4,2009-01-01 00:50:00,JENA_001,996.51,-8.27,94.1


In [21]:
df = df[
    [
        "timestamp",
        "station_id",
        "temperature_c",
        "pressure_hpa",
        "relative_humidity_pct"
    ]
].copy()

df = (
    df.sort_values("timestamp")
    .reset_index(drop=True)
)

df.head()

,timestamp,station_id,temperature_c,pressure_hpa,relative_humidity_pct
0,2009-01-01 00:10:00,JENA_001,-8.02,996.52,93.3
1,2009-01-01 00:20:00,JENA_001,-8.41,996.57,93.4
2,2009-01-01 00:30:00,JENA_001,-8.51,996.53,93.9
3,2009-01-01 00:40:00,JENA_001,-8.31,996.51,94.2
4,2009-01-01 00:50:00,JENA_001,-8.27,996.51,94.1


In [22]:
print("Rows:", len(df))

print(
    "Timestamp start:",
    df["timestamp"].min()
)

print(
    "Timestamp end:",
    df["timestamp"].max()
)

print("\nSensor statistics:")

df[
    [
        "temperature_c",
        "pressure_hpa",
        "relative_humidity_pct"
    ]
].describe()

Rows: 420551
Timestamp start: 2009-01-01 00:10:00
Timestamp end: 2017-01-01 00:00:00

Sensor statistics:


,temperature_c,pressure_hpa,relative_humidity_pct
count,420551.000000,420551.000000,420551.000000
mean,9.450147,989.212776,76.008259
std,8.423365,8.358481,16.476175
min,-23.010000,913.600000,12.950000
25%,3.360000,984.200000,65.210000
50%,9.420000,989.580000,79.300000
75%,15.470000,994.720000,89.400000
max,37.280000,1015.350000,100.000000


In [23]:
exact_duplicate_count = int(
    df.duplicated().sum()
)

print(
    "Exact duplicate rows:",
    exact_duplicate_count
)

Exact duplicate rows: 327


In [24]:
duplicate_timestamp_count = int(
    df.duplicated(
        subset=[
            "station_id",
            "timestamp"
        ]
    ).sum()
)

print(
    "Duplicate timestamps:",
    duplicate_timestamp_count
)

Duplicate timestamps: 327


In [25]:
duplicate_timestamp_rows = df[
    df.duplicated(
        subset=[
            "station_id",
            "timestamp"
        ],
        keep=False
    )
]

duplicate_timestamp_rows.head(20)

,timestamp,station_id,temperature_c,pressure_hpa,relative_humidity_pct
78622,2010-07-01 00:10:00,JENA_001,17.87,992.06,78.4
78623,2010-07-01 00:10:00,JENA_001,17.87,992.06,78.4
78624,2010-07-01 00:20:00,JENA_001,17.82,992.02,78.5
78625,2010-07-01 00:20:00,JENA_001,17.82,992.02,78.5
78626,2010-07-01 00:30:00,JENA_001,17.92,992.04,78.3
78627,2010-07-01 00:30:00,JENA_001,17.92,992.04,78.3
78628,2010-07-01 00:40:00,JENA_001,17.82,991.96,78.4
78629,2010-07-01 00:40:00,JENA_001,17.82,991.96,78.4
78630,2010-07-01 00:50:00,JENA_001,17.54,991.90,79.5
78631,2010-07-01 00:50:00,JENA_001,17.54,991.90,79.5


In [27]:
missing_values = df.isna().sum()

missing_values

missing_percentage = (
    df.isna().mean() * 100
).round(6)

missing_percentage

timestamp                0.0
station_id               0.0
temperature_c            0.0
pressure_hpa             0.0
relative_humidity_pct    0.0
dtype: float64

In [28]:
df["interval_from_previous"] = (
    df["timestamp"].diff()
)

df[
    [
        "timestamp",
        "interval_from_previous"
    ]
].head()

,timestamp,interval_from_previous
0,2009-01-01 00:10:00,NaT
1,2009-01-01 00:20:00,0 days 00:10:00
2,2009-01-01 00:30:00,0 days 00:10:00
3,2009-01-01 00:40:00,0 days 00:10:00
4,2009-01-01 00:50:00,0 days 00:10:00


In [29]:
interval_counts = (
    df["interval_from_previous"]
    .value_counts()
    .sort_values(
        ascending=False
    )
)

interval_counts.head(10)

interval_from_previous
0 days 00:10:00    420218
0 days 00:00:00       327
0 days 00:20:00         2
0 days 00:30:00         1
0 days 16:00:00         1
3 days 02:20:00         1
Name: count, dtype: int64

In [30]:
valid_intervals = (
    df["interval_from_previous"]
    .dropna()
)

if len(valid_intervals) > 0:
    expected_interval = (
        valid_intervals.mode().iloc[0]
    )
else:
    expected_interval = None

print(
    "Expected interval:",
    expected_interval
)

Expected interval: 0 days 00:10:00


In [31]:
if expected_interval is not None:

    interval_gap_mask = (
        df["interval_from_previous"]
        > expected_interval
    )

    interval_gap_count = int(
        interval_gap_mask.sum()
    )

else:

    interval_gap_mask = pd.Series(
        False,
        index=df.index
    )

    interval_gap_count = 0


print(
    "Number of interval gaps:",
    interval_gap_count
)

Number of interval gaps: 5


In [33]:
gap_rows = df.loc[
    interval_gap_mask,
    [
        "timestamp",
        "interval_from_previous"
    ]
]

gap_rows.head(20)

,timestamp,interval_from_previous
40378,2009-10-08 10:10:00,0 days 00:30:00
230019,2013-05-16 09:10:00,0 days 00:20:00
293556,2014-07-30 08:20:00,0 days 00:20:00
301673,2014-09-25 09:00:00,0 days 16:00:00
411267,2016-10-28 12:50:00,3 days 02:20:00


In [34]:
missing_observations_from_gaps = 0

if expected_interval is not None:

    for interval in valid_intervals:

        if interval > expected_interval:

            estimated_steps = (
                interval
                / expected_interval
            )

            missing_observations_from_gaps += max(
                int(round(estimated_steps)) - 1,
                0
            )

print(
    "Estimated missing observations from gaps:",
    missing_observations_from_gaps
)

Estimated missing observations from gaps: 544


In [36]:
humidity_suspect_mask = (
    (df["relative_humidity_pct"] < 0)
    |
    (df["relative_humidity_pct"] > 100)
)

humidity_suspect_count = int(
    humidity_suspect_mask.sum()
)

print(
    "Suspect humidity values:",
    humidity_suspect_count
)

df.loc[
    humidity_suspect_mask,
    [
        "timestamp",
        "relative_humidity_pct"
    ]
].head(20)

Suspect humidity values: 0


,timestamp,relative_humidity_pct


In [37]:
temperature_suspect_mask = (
    (df["temperature_c"] < -90)
    |
    (df["temperature_c"] > 60)
)

temperature_suspect_count = int(
    temperature_suspect_mask.sum()
)

print(
    "Suspect temperature values:",
    temperature_suspect_count
)

Suspect temperature values: 0


In [38]:
pressure_suspect_mask = (
    (df["pressure_hpa"] < 800)
    |
    (df["pressure_hpa"] > 1100)
)

pressure_suspect_count = int(
    pressure_suspect_mask.sum()
)

print(
    "Suspect pressure values:",
    pressure_suspect_count
)

Suspect pressure values: 0


In [39]:
suspect_any_mask = (
    temperature_suspect_mask
    |
    pressure_suspect_mask
    |
    humidity_suspect_mask
)

suspect_rows = df.loc[
    suspect_any_mask,
    [
        "timestamp",
        "temperature_c",
        "pressure_hpa",
        "relative_humidity_pct"
    ]
]

print(
    "Rows containing at least one suspect value:",
    len(suspect_rows)
)

suspect_rows.head(30)

Rows containing at least one suspect value: 0


,timestamp,temperature_c,pressure_hpa,relative_humidity_pct


In [40]:
df["temperature_unchanged"] = (
    df["temperature_c"]
    ==
    df["temperature_c"].shift(1)
)

df["pressure_unchanged"] = (
    df["pressure_hpa"]
    ==
    df["pressure_hpa"].shift(1)
)

df["humidity_unchanged"] = (
    df["relative_humidity_pct"]
    ==
    df["relative_humidity_pct"].shift(1)
)

print(
    "Consecutive unchanged temperature values:",
    int(df["temperature_unchanged"].sum())
)

print(
    "Consecutive unchanged pressure values:",
    int(df["pressure_unchanged"].sum())
)

print(
    "Consecutive unchanged humidity values:",
    int(df["humidity_unchanged"].sum())
)

Consecutive unchanged temperature values: 17915
Consecutive unchanged pressure values: 30047
Consecutive unchanged humidity values: 27784


In [41]:
station_metadata = pd.DataFrame(
    [
        {
            "station_id": "JENA_001",
            "station_name": "Jena Climate Station",
            "dataset_name": "Jena Climate 2009-2016",
            "parameters_used": (
                "temperature_c,"
                "pressure_hpa,"
                "relative_humidity_pct"
            )
        }
    ]
)

station_metadata

,station_id,station_name,dataset_name,parameters_used
0,JENA_001,Jena Climate Station,Jena Climate 2009-2016,"temperature_c,pressure_hpa,relative_humidity_pct"


In [42]:
station_metadata.to_csv(
    STATION_METADATA_FILE,
    index=False
)

print(
    "Station metadata saved to:"
)

print(
    STATION_METADATA_FILE
)

Station metadata saved to:
C:\Anomaly_Detection\data\station_metadata.csv


In [43]:
observations_df = df[
    [
        "timestamp",
        "station_id",
        "temperature_c",
        "pressure_hpa",
        "relative_humidity_pct"
    ]
].copy()

observations_df.head()

,timestamp,station_id,temperature_c,pressure_hpa,relative_humidity_pct
0,2009-01-01 00:10:00,JENA_001,-8.02,996.52,93.3
1,2009-01-01 00:20:00,JENA_001,-8.41,996.57,93.4
2,2009-01-01 00:30:00,JENA_001,-8.51,996.53,93.9
3,2009-01-01 00:40:00,JENA_001,-8.31,996.51,94.2
4,2009-01-01 00:50:00,JENA_001,-8.27,996.51,94.1


In [44]:
expected_columns = [
    "timestamp",
    "station_id",
    "temperature_c",
    "pressure_hpa",
    "relative_humidity_pct"
]

assert (
    observations_df.columns.tolist()
    ==
    expected_columns
)

assert len(observations_df) > 0

assert (
    pd.api.types
    .is_datetime64_any_dtype(
        observations_df["timestamp"]
    )
)

for column in [
    "temperature_c",
    "pressure_hpa",
    "relative_humidity_pct"
]:

    assert (
        pd.api.types
        .is_numeric_dtype(
            observations_df[column]
        )
    )

print(
    "Final normalized schema validation passed."
)

Final normalized schema validation passed.


In [45]:
observations_df.to_parquet(
    PARQUET_FILE,
    index=False,
    engine="pyarrow"
)

print(
    "Normalized observations saved to:"
)

print(
    PARQUET_FILE
)

Normalized observations saved to:
C:\Anomaly_Detection\data\processed\observations.parquet


In [46]:
quality_report = {
    "dataset": (
        "Jena Climate 2009-2016"
    ),

    "station_id": (
        "JENA_001"
    ),

    "row_count": int(
        len(observations_df)
    ),

    "timestamp_range": {
        "start": (
            observations_df[
                "timestamp"
            ]
            .min()
            .isoformat()
            if observations_df[
                "timestamp"
            ].notna().any()
            else None
        ),

        "end": (
            observations_df[
                "timestamp"
            ]
            .max()
            .isoformat()
            if observations_df[
                "timestamp"
            ].notna().any()
            else None
        )
    },

    "duplicates": {
        "exact_duplicate_rows":
            exact_duplicate_count,

        "duplicate_timestamps":
            duplicate_timestamp_count
    },

    "missing_values": {
        column: int(value)

        for column, value
        in observations_df
        .isna()
        .sum()
        .items()
    },

    "sampling": {
        "expected_interval":
            str(expected_interval)
            if expected_interval
            is not None
            else None,

        "interval_gap_count":
            interval_gap_count,

        "estimated_missing_observations":
            int(
                missing_observations_from_gaps
            )
    },

    "suspect_values": {
        "temperature":
            temperature_suspect_count,

        "pressure":
            pressure_suspect_count,

        "relative_humidity":
            humidity_suspect_count,

        "rows_with_any_suspect_value":
            int(len(suspect_rows))
    },

    "basic_frozen_value_audit": {
        "temperature_consecutive_equal":
            int(
                df[
                    "temperature_unchanged"
                ].sum()
            ),

        "pressure_consecutive_equal":
            int(
                df[
                    "pressure_unchanged"
                ].sum()
            ),

        "humidity_consecutive_equal":
            int(
                df[
                    "humidity_unchanged"
                ].sum()
            )
    }
}

quality_report

{'dataset': 'Jena Climate 2009-2016',
 'station_id': 'JENA_001',
 'row_count': 420551,
 'timestamp_range': {'start': '2009-01-01T00:10:00',
  'end': '2017-01-01T00:00:00'},
 'duplicates': {'exact_duplicate_rows': 327, 'duplicate_timestamps': 327},
 'missing_values': {'timestamp': 0,
  'station_id': 0,
  'temperature_c': 0,
  'pressure_hpa': 0,
  'relative_humidity_pct': 0},
 'sampling': {'expected_interval': '0 days 00:10:00',
  'interval_gap_count': 5,
  'estimated_missing_observations': 544},
 'suspect_values': {'temperature': 0,
  'pressure': 0,
  'relative_humidity': 0,
  'rows_with_any_suspect_value': 0},
 'basic_frozen_value_audit': {'temperature_consecutive_equal': 17915,
  'pressure_consecutive_equal': 30047,
  'humidity_consecutive_equal': 27784}}

In [47]:
with open(
    QUALITY_REPORT_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        quality_report,
        file,
        indent=4
    )

print(
    "Quality report saved to:"
)

print(
    QUALITY_REPORT_FILE
)

Quality report saved to:
C:\Anomaly_Detection\data\processed\data_quality_report.json


In [48]:
required_final_columns = {
    "timestamp",
    "station_id",
    "temperature_c",
    "pressure_hpa",
    "relative_humidity_pct"
}

assert (
    set(observations_df.columns)
    ==
    required_final_columns
)

print(
    "TEST PASSED: Required columns"
)

TEST PASSED: Required columns


In [49]:
assert (
    len(observations_df)
    ==
    len(raw_df)
)

print(
    "TEST PASSED: Row count preserved"
)

TEST PASSED: Row count preserved


In [50]:
assert (
    observations_df[
        "station_id"
    ]
    .eq("JENA_001")
    .all()
)

print(
    "TEST PASSED: Station ID"
)

TEST PASSED: Station ID


In [51]:
assert (
    pd.api.types
    .is_datetime64_any_dtype(
        observations_df[
            "timestamp"
        ]
    )
)

print(
    "TEST PASSED: Timestamp datatype"
)

TEST PASSED: Timestamp datatype


In [52]:
for column in [
    "temperature_c",
    "pressure_hpa",
    "relative_humidity_pct"
]:

    assert (
        pd.api.types
        .is_numeric_dtype(
            observations_df[column]
        )
    )

print(
    "TEST PASSED: Sensor datatypes"
)

TEST PASSED: Sensor datatypes


In [54]:
assert (
    observations_df[
        "timestamp"
    ].is_monotonic_increasing
)

print(
    "TEST PASSED: Chronological ordering"
)

TEST PASSED: Chronological ordering


In [55]:
assert PARQUET_FILE.exists()

print(
    "TEST PASSED: "
    "observations.parquet exists"
)

TEST PASSED: observations.parquet exists


In [56]:
assert (
    QUALITY_REPORT_FILE.exists()
)

print(
    "TEST PASSED: "
    "data_quality_report.json exists"
)

TEST PASSED: data_quality_report.json exists


In [57]:
assert (
    STATION_METADATA_FILE.exists()
)

print(
    "TEST PASSED: "
    "station_metadata.csv exists"
)

TEST PASSED: station_metadata.csv exists


In [58]:
verification_df = pd.read_parquet(
    PARQUET_FILE
)

verification_df.head()

,timestamp,station_id,temperature_c,pressure_hpa,relative_humidity_pct
0,2009-01-01 00:10:00,JENA_001,-8.02,996.52,93.3
1,2009-01-01 00:20:00,JENA_001,-8.41,996.57,93.4
2,2009-01-01 00:30:00,JENA_001,-8.51,996.53,93.9
3,2009-01-01 00:40:00,JENA_001,-8.31,996.51,94.2
4,2009-01-01 00:50:00,JENA_001,-8.27,996.51,94.1


In [59]:
assert (
    len(verification_df)
    ==
    len(observations_df)
)

print(
    "TEST PASSED: "
    "Persisted Parquet row count"
)

TEST PASSED: Persisted Parquet row count


In [60]:
assert (
    verification_df.columns.tolist()
    ==
    observations_df.columns.tolist()
)

print(
    "TEST PASSED: "
    "Persisted Parquet schema"
)

TEST PASSED: Persisted Parquet schema


In [63]:
print(f"Total Rows                    : {len(observations_df):,}")
print(f"Timestamp Start               : {observations_df['timestamp'].min()}")
print(f"Timestamp End                 : {observations_df['timestamp'].max()}")
print(f"Sampling Interval             : {expected_interval}")

print(f"Exact Duplicate Rows          : {exact_duplicate_count:,}")
print(f"Duplicate Timestamps          : {duplicate_timestamp_count:,}")

print(f"Missing Timestamps            : {observations_df['timestamp'].isna().sum():,}")
print(f"Missing Temperature Values    : {observations_df['temperature_c'].isna().sum():,}")
print(f"Missing Pressure Values       : {observations_df['pressure_hpa'].isna().sum():,}")
print(f"Missing Humidity Values       : {observations_df['relative_humidity_pct'].isna().sum():,}")

print(f"Interval Gaps                 : {interval_gap_count:,}")
print(f"Estimated Missing Observations: {missing_observations_from_gaps:,}")

print(f"Suspect Temperature Values    : {temperature_suspect_count:,}")
print(f"Suspect Pressure Values       : {pressure_suspect_count:,}")
print(f"Suspect Humidity Values       : {humidity_suspect_count:,}")

print(f"Unchanged Temperature Values  : {int(df['temperature_unchanged'].sum()):,}")
print(f"Unchanged Pressure Values     : {int(df['pressure_unchanged'].sum()):,}")
print(f"Unchanged Humidity Values     : {int(df['humidity_unchanged'].sum()):,}")

Total Rows                    : 420,551
Timestamp Start               : 2009-01-01 00:10:00
Timestamp End                 : 2017-01-01 00:00:00
Sampling Interval             : 0 days 00:10:00
Exact Duplicate Rows          : 327
Duplicate Timestamps          : 327
Missing Timestamps            : 0
Missing Temperature Values    : 0
Missing Pressure Values       : 0
Missing Humidity Values       : 0
Interval Gaps                 : 5
Estimated Missing Observations: 544
Suspect Temperature Values    : 0
Suspect Pressure Values       : 0
Suspect Humidity Values       : 0
Unchanged Temperature Values  : 17,915
Unchanged Pressure Values     : 30,047
Unchanged Humidity Values     : 27,784


In [64]:
print(
    "Generated Phase 1 files:\n"
)

print(
    f"1. {PARQUET_FILE}"
)

print(
    f"2. {QUALITY_REPORT_FILE}"
)

print(
    f"3. {STATION_METADATA_FILE}"
)

print()

print(
    "Parquet exists:",
    PARQUET_FILE.exists()
)

print(
    "Quality report exists:",
    QUALITY_REPORT_FILE.exists()
)

print(
    "Station metadata exists:",
    STATION_METADATA_FILE.exists()
)

Generated Phase 1 files:

1. C:\Anomaly_Detection\data\processed\observations.parquet
2. C:\Anomaly_Detection\data\processed\data_quality_report.json
3. C:\Anomaly_Detection\data\station_metadata.csv

Parquet exists: True
Quality report exists: True
Station metadata exists: True
